## google/paligemma2-3b-mix-224  
    Final Overall Accuracy: 67.59%  
    Per-task Accuracy:  
    Count: 60.66% (478/788)  
    Relation: 76.00% (494/650)


In [1]:
from src.models import DinoVLM
import torch
from icecream import ic
import json
from transformers import pipeline
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from src.dataset_utils import *
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import os
from tqdm.notebook import tqdm

Failed to load /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/system/miniconda3/envs/gemma/lib/python3.10/site-packages/torchao/_C_cutlass_90a.abi3.so


In [2]:
model = DinoVLM()
processor = model.processor
model.freeze_lm()
model.freeze_vision()

total_trainable_params = sum([p.numel() for p in model.parameters() if p.requires_grad])
print(f'[INFO] Total trainable parameters: {total_trainable_params}')

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using cache found in /home/system/.cache/torch/hub/ywyue_FiT3D_main
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[INFO] Freezed LM
[INFO] Freezed ViT
[INFO] Total trainable parameters: 887040


In [3]:
# def collate_fn(examples):
#     texts = []
#     images = []
#     for image, text in zip(examples['images'], examples['texts']):
#         for t in text: # its a list of dict
#             if image.mode != "RGB":
#                 image = image.convert("RGB")
#             user, assistant = t['user'], t['assistant']
#             texts.append((user, assistant))
#             images.append(image)
        
#         prompt_texts = [f"{u}\n" for u, _ in texts] # only user question
#         full_texts   = [f"{u}\n{a}" for u, a in texts] # user question + label

#         batch_input = processor(
#             text=full_texts,
#             images=images,
#             return_tensors="pt",
#             padding=True,
#         )

#         prompt_only = processor(
#             text=prompt_texts,
#             images=images,
#             return_tensors="pt",
#             padding=True,
#         )

#         labels = batch_input["input_ids"].clone()

#         labels[labels == processor.tokenizer.pad_token_id] = -100
#         labels[labels == processor.tokenizer.convert_tokens_to_ids("<image>")] = -100

#         for i in range(len(texts)):
#             prompt_len = prompt_only["input_ids"][i].shape[0]
#             labels[i, :prompt_len] = -100

#         batch_input["labels"] = labels
#         return batch_input


In [4]:

def collate_fn(examples):
    texts = []
    images = []

    for ex in examples:
        image = ex["images"][0]

        if image.mode != "RGB":
            image = image.convert("RGB")

        image = image.resize((384, 384))

        qa = random.choice(ex["texts"])

        user = qa["user"]
        assistant = qa["assistant"]

        texts.append((user, assistant))
        images.append(image)

    prompt_texts = [f"{u}\n" for u, _ in texts]
    full_texts   = [f"{u}\n{a}" for u, a in texts]

    batch_input = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    prompt_only = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    labels = batch_input["input_ids"].clone()

    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == processor.tokenizer.convert_tokens_to_ids("<image>")] = -100

    for i in range(len(texts)):
        prompt_len = prompt_only["input_ids"][i].shape[0]
        labels[i, :prompt_len] = -100

    batch_input["labels"] = labels
    return batch_input

In [5]:
dataset = load_dataset("HuggingFaceM4/the_cauldron", "vqav2", split="train")
# dataset['train'][0]
dataset_split = dataset.train_test_split(test_size = 0.1)

train_dataloader = DataLoader(dataset_split['train'], batch_size=1, shuffle = True, collate_fn=collate_fn)
test_dataset = DataLoader(dataset_split['test'], batch_size=1, shuffle = True, collate_fn=collate_fn)


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/27 [00:00<?, ?it/s]

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=r"language_model\.layers\.\d+\.self_attn\.(q|k|v)_proj",
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,579,328 || all params: 2,644,666,112 || trainable%: 0.1732


In [7]:
for p in model.connector.parameters():
    p.requires_grad = True

In [12]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

4579328

In [8]:
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name)

base_model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.language_model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.language_model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.language_model.layers.1.self_attn.q_proj.lora_A.default.weight
base_model.model.language_model.layers.1.self_attn.q_proj.lora_B.default.weight
base_model.model.language_model.layers.1.self_attn.k_proj.lora_A.default.weight
base_model.model.language_model.layers.1.self_attn.k_proj.lora_B.default.weight
base_model.model.language_model.layers.1.self_attn.v_proj.lora_A.default.weight
base_model.model.language_model.layers.1.self_attn.v_proj.lora_B.default.weight
base_model.model.language_model.layers.2

In [9]:
NUM_EPOCHS      = 4
GRAD_ACCUM_STEPS = 4
LR              = 2e-4
MAX_GRAD_NORM   = 1.0
SAVE_DIR        = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)
# scheduler = CosineAnnealingLR(optimizer, T_max=len(train_dataloader) * NUM_EPOCHS)
# scaler    = torch.cuda.amp.GradScaler()

# best_eval_loss = float("inf")

In [10]:
trainable = {id(p): name for name, p in model.named_parameters() if p.requires_grad}

for group in optimizer.param_groups:
    for p in group['params']:
        print(trainable.get(id(p), "unknown"), p.shape)

base_model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([16, 2304])
base_model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([2048, 16])
base_model.model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight torch.Size([16, 2304])
base_model.model.language_model.layers.0.self_attn.k_proj.lora_B.default.weight torch.Size([1024, 16])
base_model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([16, 2304])
base_model.model.language_model.layers.0.self_attn.v_proj.lora_B.default.weight torch.Size([1024, 16])
base_model.model.language_model.layers.1.self_attn.q_proj.lora_A.default.weight torch.Size([16, 2304])
base_model.model.language_model.layers.1.self_attn.q_proj.lora_B.default.weight torch.Size([2048, 16])
base_model.model.language_model.layers.1.self_attn.k_proj.lora_A.default.weight torch.Size([16, 2304])
base_model.model.language_model.layers.1.self_attn.k_proj.lora_B.default.

In [8]:
device = next(model.parameters()).device
print(f"Model is on device: {device}")

Model is on device: cuda:0


In [9]:

# try:
#     for b in train_dataloader:
#         # try:
#             for key, data in b.items():
#                 # print(key, data.shape)
#                 if data.shape[1] <3:
#                     print(key, data.shape)
# except Exception as err:
#     print(err)
#     # print(b['attention_mask'])
#     # print(b['input_ids'])
#     # break

In [10]:
import random

def collate_fn(examples):
    texts = []
    images = []

    for ex in examples:
        image = ex["images"][0]

        if image.mode != "RGB":
            image = image.convert("RGB")

        image = image.resize((384, 384))

        qa = random.choice(ex["texts"])

        user = qa["user"]
        assistant = qa["assistant"]

        texts.append((user, assistant))
        images.append(image)

    prompt_texts = [f"{u}\n" for u, _ in texts]
    full_texts   = [f"{u}\n{a}" for u, a in texts]

    batch_input = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    prompt_only = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    labels = batch_input["input_ids"].clone()

    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == processor.tokenizer.convert_tokens_to_ids("<image>")] = -100

    for i in range(len(texts)):
        prompt_len = prompt_only["input_ids"][i].shape[0]
        labels[i, :prompt_len] = -100

    batch_input["labels"] = labels
    return batch_input

In [ ]:
import logging

logging.getLogger("transformers.models.paligemma.processing_paligemma").setLevel(logging.ERROR)

: 

In [ ]:

device = 'cuda'
import warnings

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    train_loss   = 0.0
    optimizer.zero_grad()

    for step, batch in tqdm(enumerate(train_dataloader), leave = True, total = len(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}

        warnings.filterwarnings(
            "ignore",
            message="You are passing both `text` and `images` to `PaliGemmaProcessor`.*"
        )
        with torch.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(**batch)
            loss    = outputs.loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        train_loss += outputs.loss.item()  # log unscaled loss

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if step % 10 == 0:
            avg = train_loss / (step + 1)
            # print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Step {step}/{len(train_dataloader)} | Loss {avg:.4f}")

    # handle leftover steps if dataset isn't divisible by GRAD_ACCUM_STEPS
    if (step + 1) % GRAD_ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} complete | Avg train loss: {avg_train_loss:.4f}")

    model.eval()
    eval_loss = 0.0

    with torch.no_grad():
        for batch in test_dataset:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
            eval_loss += outputs.loss.item()
        
    avg_eval_loss = eval_loss / len(test_dataset)
    print(f"Epoch {epoch+1} | Eval loss: {avg_eval_loss:.4f}")

    if avg_eval_loss < best_eval_loss:
        best_eval_loss = avg_eval_loss
        # saves only LoRA + connector weights, not the frozen LM/ViT
        model.save_pretrained(os.path.join(SAVE_DIR, "best"))
        print(f"  ↳ New best saved (eval_loss={avg_eval_loss:.4f})")

    # save every epoch as a checkpoint too
    model.save_pretrained(os.path.join(SAVE_DIR, f"epoch_{epoch+1}"))

print(f"\nTraining done. Best eval loss: {best_eval_loss:.4f}")

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/74494 [00:00<?, ?it/s]


Epoch 1 complete | Avg train loss: 0.8348
